# Walmart Sales Forecasting — EDA & Feature Engineering

**Kaggle Competition:** Walmart Recruiting - Store Sales Forecasting  
**ამოცანა:** 45 მაღაზიის ყოველკვირეული გაყიდვების პროგნოზი  
**მეტრიკა:** Weighted MAE (სადღესასწაულო კვირებს 5x წონა)

ეს notebook მოიცავს:
1. მონაცემების ჩამოტვირთვა და ჩატვირთვა
2. Exploratory Data Analysis (EDA)
3. Feature Engineering
4. დამუშავებული მონაცემების შენახვა Drive-ზე

## 1. Setup და მონაცემების ჩამოტვირთვა

In [ ]:
# ბიბლიოთეკების ინსტალაცია
!pip install kaggle --quiet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 12

print(" ბიბლიოთეკები მზადაა")

In [ ]:
# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/walmart'
DATA_DIR = f'{PROJECT_DIR}/data'
os.makedirs(DATA_DIR, exist_ok=True)
print(f" პროექტის ფოლდერი: {PROJECT_DIR}")

In [ ]:
import os
os.environ['KAGGLE_USERNAME'] = 'zaqariaberidze'
os.environ['KAGGLE_KEY'] = '76fd1c2dc0063334fab9c653ef331af9'

In [ ]:
# მონაცემების ჩამოტვირთვა
!kaggle competitions download -c walmart-recruiting-store-sales-forecasting -p {DATA_DIR}
!unzip -o {DATA_DIR}/walmart-recruiting-store-sales-forecasting.zip -d {DATA_DIR}
!ls {DATA_DIR}

## 2. მონაცემების ჩატვირთვა

In [ ]:
# მონაცემების წაკითხვა
train = pd.read_csv(f'{DATA_DIR}/train.csv.zip')
test = pd.read_csv(f'{DATA_DIR}/test.csv.zip')
stores = pd.read_csv(f'{DATA_DIR}/stores.csv')
features = pd.read_csv(f'{DATA_DIR}/features.csv.zip')

# Date სვეტის datetime-ად გადაყვანა
for df in [train, test, features]:
    df['Date'] = pd.to_datetime(df['Date'])

print(f"Train shape: {train.shape}")
print(f"Test shape:  {test.shape}")
print(f"Stores shape: {stores.shape}")
print(f"Features shape: {features.shape}")

In [ ]:
# თითოეული DataFrame-ის პირველი სტრიქონები
print("=== TRAIN ===")
display(train.head())
print("\n=== TEST ===")
display(test.head())
print("\n=== STORES ===")
display(stores.head())
print("\n=== FEATURES ===")
display(features.head())

In [ ]:
# მონაცემთა ტიპები და ინფორმაცია
print("=== TRAIN INFO ===")
print(train.dtypes)
print(f"\nDate range: {train['Date'].min()} → {train['Date'].max()}")
print(f"Stores: {train['Store'].nunique()}")
print(f"Departments: {train['Dept'].nunique()}")
print(f"\n=== TEST INFO ===")
print(f"Date range: {test['Date'].min()} → {test['Date'].max()}")

## 3. Exploratory Data Analysis (EDA)

### 3.1 Missing Values

In [ ]:
# Missing values ანალიზი
print("=== TRAIN Missing Values ===")
print(train.isnull().sum())

print("\n=== FEATURES Missing Values ===")
print(features.isnull().sum())
print(f"\nFeatures-ში სულ: {features.shape[0]} სტრიქონი")
print(f"MarkDown1 არ არის: {features['MarkDown1'].isnull().sum()} ({features['MarkDown1'].isnull().mean()*100:.1f}%)")
print(f"MarkDown2 არ არის: {features['MarkDown2'].isnull().sum()} ({features['MarkDown2'].isnull().mean()*100:.1f}%)")
print(f"MarkDown3 არ არის: {features['MarkDown3'].isnull().sum()} ({features['MarkDown3'].isnull().mean()*100:.1f}%)")
print(f"MarkDown4 არ არის: {features['MarkDown4'].isnull().sum()} ({features['MarkDown4'].isnull().mean()*100:.1f}%)")
print(f"MarkDown5 არ არის: {features['MarkDown5'].isnull().sum()} ({features['MarkDown5'].isnull().mean()*100:.1f}%)")
print(f"CPI არ არის: {features['CPI'].isnull().sum()} ({features['CPI'].isnull().mean()*100:.1f}%)")
print(f"Unemployment არ არის: {features['Unemployment'].isnull().sum()} ({features['Unemployment'].isnull().mean()*100:.1f}%)")

### 3.2 Weekly Sales — განაწილება

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ჰისტოგრამა
axes[0].hist(train['Weekly_Sales'], bins=100, edgecolor='black', alpha=0.7)
axes[0].set_title('Weekly Sales Distribution')
axes[0].set_xlabel('Weekly Sales')
axes[0].set_ylabel('Frequency')

# უარყოფითი გაყიდვები
negative_sales = train[train['Weekly_Sales'] < 0]
axes[1].hist(negative_sales['Weekly_Sales'], bins=50, edgecolor='black', alpha=0.7, color='red')
axes[1].set_title(f'Negative Sales ({len(negative_sales)} rows, {len(negative_sales)/len(train)*100:.2f}%)')
axes[1].set_xlabel('Weekly Sales')

# Box plot (outliers გარეშე ჩანს pattern)
axes[2].boxplot(train['Weekly_Sales'], vert=True)
axes[2].set_title('Weekly Sales Box Plot')

plt.tight_layout()
plt.show()

print(f"Weekly Sales Stats:")
print(train['Weekly_Sales'].describe())

### 3.3 დროითი ტრენდები

In [ ]:
# საშუალო გაყიდვები კვირის მიხედვით
weekly_avg = train.groupby('Date')['Weekly_Sales'].mean().reset_index()

plt.figure(figsize=(16, 6))
plt.plot(weekly_avg['Date'], weekly_avg['Weekly_Sales'], linewidth=1)
plt.title('Average Weekly Sales Over Time')
plt.xlabel('Date')
plt.ylabel('Average Weekly Sales')
plt.axhline(y=weekly_avg['Weekly_Sales'].mean(), color='red', linestyle='--', alpha=0.5, label='Overall Mean')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# სეზონურობა — თვის მიხედვით
train_temp = train.copy()
train_temp['Month'] = train_temp['Date'].dt.month
train_temp['Year'] = train_temp['Date'].dt.year

monthly_sales = train_temp.groupby(['Year', 'Month'])['Weekly_Sales'].mean().reset_index()

plt.figure(figsize=(14, 6))
for year in sorted(monthly_sales['Year'].unique()):
    year_data = monthly_sales[monthly_sales['Year'] == year]
    plt.plot(year_data['Month'], year_data['Weekly_Sales'], marker='o', label=str(year))

plt.title('Monthly Average Sales by Year (Seasonality)')
plt.xlabel('Month')
plt.ylabel('Average Weekly Sales')
plt.xticks(range(1, 13))
plt.legend()
plt.tight_layout()
plt.show()

### 3.4 Holiday Impact

In [ ]:
# სადღესასწაულო vs ჩვეულებრივი კვირები
holiday_sales = train.groupby('IsHoliday')['Weekly_Sales'].agg(['mean', 'median', 'std', 'count'])
print("Holiday vs Non-Holiday Sales:")
display(holiday_sales)

fig, ax = plt.subplots(figsize=(8, 5))
train.groupby('IsHoliday')['Weekly_Sales'].mean().plot(kind='bar', ax=ax, color=['steelblue', 'coral'])
ax.set_title('Average Weekly Sales: Holiday vs Non-Holiday')
ax.set_xticklabels(['Non-Holiday', 'Holiday'], rotation=0)
ax.set_ylabel('Average Weekly Sales')
plt.tight_layout()
plt.show()

### 3.5 Store Analysis

In [ ]:
# მაღაზიის ტიპის მიხედვით
train_stores = train.merge(stores, on='Store', how='left')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ტიპის მიხედვით
train_stores.groupby('Type')['Weekly_Sales'].mean().plot(kind='bar', ax=axes[0], color=['steelblue', 'coral', 'green'])
axes[0].set_title('Avg Weekly Sales by Store Type')
axes[0].set_xticklabels(['A', 'B', 'C'], rotation=0)

# ზომის მიხედვით
axes[1].scatter(train_stores.groupby('Store')['Weekly_Sales'].mean().values,
                stores.set_index('Store').loc[train_stores.groupby('Store')['Weekly_Sales'].mean().index, 'Size'].values,
                alpha=0.6)
axes[1].set_xlabel('Average Weekly Sales')
axes[1].set_ylabel('Store Size')
axes[1].set_title('Store Size vs Average Sales')

plt.tight_layout()
plt.show()

In [ ]:
# დეპარტამენტების ანალიზი
dept_sales = train.groupby('Dept')['Weekly_Sales'].mean().sort_values(ascending=False)

plt.figure(figsize=(16, 6))
dept_sales.head(20).plot(kind='bar', color='steelblue')
plt.title('Top 20 Departments by Average Weekly Sales')
plt.xlabel('Department')
plt.ylabel('Average Weekly Sales')
plt.tight_layout()
plt.show()

print(f"სულ დეპარტამენტი: {train['Dept'].nunique()}")
print(f"\nTop 5 დეპარტამენტი:")
print(dept_sales.head())

### 3.6 Features Dataset Analysis

In [ ]:
# features dataset-ის ვიზუალიზაცია
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Temperature
features.groupby('Date')['Temperature'].mean().plot(ax=axes[0,0], title='Average Temperature Over Time')

# Fuel Price
features.groupby('Date')['Fuel_Price'].mean().plot(ax=axes[0,1], title='Average Fuel Price Over Time')

# CPI
features.groupby('Date')['CPI'].mean().plot(ax=axes[1,0], title='Average CPI Over Time')

# Unemployment
features.groupby('Date')['Unemployment'].mean().plot(ax=axes[1,1], title='Average Unemployment Over Time')

plt.tight_layout()
plt.show()

In [ ]:
# კორელაციის ანალიზი
# ჯერ train-ს features-თან ვაერთიანებთ
train_full = train.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
train_full = train_full.merge(stores, on='Store', how='left')

numeric_cols = ['Weekly_Sales', 'Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Size',
                'MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

plt.figure(figsize=(12, 8))
corr_matrix = train_full[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 4. Feature Engineering

ახლა შევქმნით ახალ ფიჩერებს, რომლებსაც ყველა მოდელი გამოიყენებს.

**შენიშვნა:** Lag ფიჩერები და rolling statistics მოდელ-სპეციფიკურია.  
Tree-based მოდელები (XGBoost, LightGBM) იყენებენ, DL მოდელები (N-BEATS, PatchTST) — არა.  
ამიტომ ეს ფიჩერები მოდელის notebook-ებში დაემატება.

In [ ]:
def create_features(df):
    """ძირითადი Feature Engineering — ორივე train და test-ისთვის."""
    df = df.copy()

    # ===== დროითი ფიჩერები =====
    df['Year'] = df['Date'].dt.year
    df['Month'] = df['Date'].dt.month
    df['Week'] = df['Date'].dt.isocalendar().week.astype(int)
    df['Day_of_Year'] = df['Date'].dt.dayofyear
    df['Quarter'] = df['Date'].dt.quarter

    # ===== კონკრეტული დღესასწაულების იდენტიფიკაცია =====
    # Walmart-ის მნიშვნელოვანი დღესასწაულები:
    # Super Bowl: Feb 12, 2010; Feb 11, 2011; Feb 10, 2012; Feb 8, 2013
    # Labor Day: Sep 10, 2010; Sep 9, 2011; Sep 7, 2012; Sep 6, 2013
    # Thanksgiving: Nov 26, 2010; Nov 25, 2011; Nov 23, 2012; Nov 29, 2013
    # Christmas: Dec 31, 2010; Dec 30, 2011; Dec 28, 2012; Dec 27, 2013

    super_bowl = pd.to_datetime(['2010-02-12', '2011-02-11', '2012-02-10', '2013-02-08'])
    labor_day = pd.to_datetime(['2010-09-10', '2011-09-09', '2012-09-07', '2013-09-06'])
    thanksgiving = pd.to_datetime(['2010-11-26', '2011-11-25', '2012-11-23', '2013-11-29'])
    christmas = pd.to_datetime(['2010-12-31', '2011-12-30', '2012-12-28', '2013-12-27'])

    df['Is_SuperBowl'] = df['Date'].isin(super_bowl).astype(int)
    df['Is_LaborDay'] = df['Date'].isin(labor_day).astype(int)
    df['Is_Thanksgiving'] = df['Date'].isin(thanksgiving).astype(int)
    df['Is_Christmas'] = df['Date'].isin(christmas).astype(int)

    # ===== ციკლური encoding თვისთვის =====
    df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
    df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
    df['Week_Sin'] = np.sin(2 * np.pi * df['Week'] / 52)
    df['Week_Cos'] = np.cos(2 * np.pi * df['Week'] / 52)

    return df

print(" Feature engineering ფუნქცია მზადაა")

In [ ]:
# ===== მონაცემების გაერთიანება =====

# Train
train_merged = train.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
train_merged = train_merged.merge(stores, on='Store', how='left')

# Test
test_merged = test.merge(features, on=['Store', 'Date', 'IsHoliday'], how='left')
test_merged = test_merged.merge(stores, on='Store', how='left')

print(f"Train merged shape: {train_merged.shape}")
print(f"Test merged shape: {test_merged.shape}")

In [ ]:
# ===== Missing Values — MarkDown-ების შევსება =====
# MarkDown ფიჩერები NaN-ია 2011 წლის ნოემბრამდე (აქციები არ არსებობდა)
# შევავსებთ 0-ით (აქცია არ იყო = 0)

markdown_cols = ['MarkDown1', 'MarkDown2', 'MarkDown3', 'MarkDown4', 'MarkDown5']

for col in markdown_cols:
    train_merged[col] = train_merged[col].fillna(0)
    test_merged[col] = test_merged[col].fillna(0)

# CPI და Unemployment — forward fill Store-ის მიხედვით
for col in ['CPI', 'Unemployment']:
    train_merged[col] = train_merged.groupby('Store')[col].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))
    test_merged[col] = test_merged.groupby('Store')[col].transform(lambda x: x.fillna(method='ffill').fillna(method='bfill'))

print("Missing values after filling:")
print("Train:", train_merged.isnull().sum().sum())
print("Test:", test_merged.isnull().sum().sum())

In [ ]:
# ===== Feature Engineering — ფიჩერების დამატება =====
train_processed = create_features(train_merged)
test_processed = create_features(test_merged)

# Store Type encoding
type_mapping = {'A': 0, 'B': 1, 'C': 2}
train_processed['Type_Encoded'] = train_processed['Type'].map(type_mapping)
test_processed['Type_Encoded'] = test_processed['Type'].map(type_mapping)

# IsHoliday → int
train_processed['IsHoliday'] = train_processed['IsHoliday'].astype(int)
test_processed['IsHoliday'] = test_processed['IsHoliday'].astype(int)

print(f"Train processed shape: {train_processed.shape}")
print(f"Test processed shape: {test_processed.shape}")
print(f"\nColumns: {list(train_processed.columns)}")

In [ ]:
# ===== საბოლოო შემოწმება =====
print("=== Train Processed ===")
display(train_processed.head())
print(f"\nShape: {train_processed.shape}")
print(f"Missing: {train_processed.isnull().sum().sum()}")
print(f"Date range: {train_processed['Date'].min()} → {train_processed['Date'].max()}")

print("\n=== Test Processed ===")
display(test_processed.head())
print(f"\nShape: {test_processed.shape}")
print(f"Missing: {test_processed.isnull().sum().sum()}")
print(f"Date range: {test_processed['Date'].min()} → {test_processed['Date'].max()}")

In [ ]:
# ===== ახალი ფიჩერების ეფექტი =====
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Holiday type breakdown
holiday_types = ['Is_SuperBowl', 'Is_LaborDay', 'Is_Thanksgiving', 'Is_Christmas']
holiday_means = []
for h in holiday_types:
    mask = train_processed[h] == 1
    if mask.sum() > 0:
        holiday_means.append(train_processed.loc[mask, 'Weekly_Sales'].mean())
    else:
        holiday_means.append(0)

axes[0,0].bar(holiday_types, holiday_means, color=['blue', 'green', 'orange', 'red'])
axes[0,0].set_title('Avg Weekly Sales by Holiday Type')
axes[0,0].tick_params(axis='x', rotation=45)

# Monthly pattern
train_processed.groupby('Month')['Weekly_Sales'].mean().plot(kind='bar', ax=axes[0,1], color='steelblue')
axes[0,1].set_title('Avg Weekly Sales by Month')

# Store Type
train_processed.groupby('Type')['Weekly_Sales'].mean().plot(kind='bar', ax=axes[1,0], color=['steelblue', 'coral', 'green'])
axes[1,0].set_title('Avg Weekly Sales by Store Type')

# Weekly pattern
train_processed.groupby('Week')['Weekly_Sales'].mean().plot(ax=axes[1,1], color='steelblue')
axes[1,1].set_title('Avg Weekly Sales by Week of Year')
axes[1,1].set_xlabel('Week')

plt.tight_layout()
plt.show()

## 5. დამუშავებული მონაცემების შენახვა

ეს ფაილები გამოიყენება ყველა მოდელის notebook-ში.

**სტრუქტურა Drive-ზე:**
```
/content/drive/MyDrive/walmart/
├── data/
│   ├── train.csv.zip (original)
│   ├── test.csv.zip (original)
│   ├── stores.csv (original)
│   ├── features.csv (original)
│   ├── sampleSubmission.csv.zip (original)
│   ├── train_processed.csv ← ეს ვქმნით
│   └── test_processed.csv  ← ეს ვქმნით
```

In [ ]:
# შენახვა Drive-ზე
train_processed.to_csv(f'{DATA_DIR}/train_processed.csv', index=False)
test_processed.to_csv(f'{DATA_DIR}/test_processed.csv', index=False)

# ასევე stores და features ცალკე (ზოგი მოდელისთვის შეიძლება დაჭირდეს)
stores.to_csv(f'{DATA_DIR}/stores_clean.csv', index=False)

print(" ფაილები შენახულია:")
print(f"   {DATA_DIR}/train_processed.csv ({train_processed.shape})")
print(f"   {DATA_DIR}/test_processed.csv ({test_processed.shape})")
print(f"   {DATA_DIR}/stores_clean.csv ({stores.shape})")


## შემდეგი ნაბიჯი

მონაცემები მზადაა. ახლა გადავდივართ მოდელების notebook-ებზე:

**ზაქარია ბერიძე:** XGBoost, N-BEATS, ARIMA/SARIMA, PatchTST  
**გიგა ბერაძე:** LightGBM, DLinear, Prophet, TimesFM (bonus)

თითოეული notebook:
1. იტვირთავს `train_processed.csv` და `test_processed.csv`
2. ამატებს მოდელ-სპეციფიკურ ფიჩერებს (მაგ: lag ფიჩერები tree მოდელებისთვის)
3. ატრენინგებს მოდელს
4. ლოგავს MLflow/WandB-ზე
5. ინახავს საუკეთესო მოდელს Pipeline-ად